**1.1**

I chose the watershed dataset. The response variable is streamflow_index, with predictors snowpack_pct_normal. precip_mm, and mean_temp_c. The goal of this report is to fit and diagnose a linear model using matrix computation. 

In [1]:
import pandas as pd
import numpy as np 

df_watershed = pd.read_csv("../Data/watershed.csv")

In [2]:
print(df_watershed.head())
print(df_watershed.shape)
print(df_watershed.dtypes)
print(df_watershed.isna().sum())
print(df_watershed["id"].is_unique)

   id  snowpack_pct_normal  precip_mm  mean_temp_c  streamflow_index
0   1                83.41     267.74        12.88             31.06
1   2                47.93     212.76        15.71             17.33
2   3                85.17     435.74        13.15             46.78
3   4               115.00     432.22        11.46             67.16
4   5               122.18     506.05        11.14             78.78
(125, 5)
id                       int64
snowpack_pct_normal    float64
precip_mm              float64
mean_temp_c            float64
streamflow_index       float64
dtype: object
id                     0
snowpack_pct_normal    0
precip_mm              0
mean_temp_c            0
streamflow_index       0
dtype: int64
True


There are a total of 125 observations, with 5 variables. This dataframe is a 125x5 matrix. There are no missing values present, and every ID is unique. 

**Part 2**

In [3]:
predictors = df_watershed[[
    "snowpack_pct_normal",
    "precip_mm",
    "mean_temp_c"
]].to_numpy()

y = df_watershed[["streamflow_index"]].to_numpy()



In [4]:
means = predictors.mean(axis=0)
sds = predictors.std(axis=0)
Z = (
    predictors - means
) / sds

print(Z.mean(axis=0))
print(Z.std(axis=0))

[-3.47721851e-16 -6.06625861e-16  1.56319402e-16]
[1. 1. 1.]


In [5]:
X = np.column_stack([
    np.ones(len(Z)),
    Z
])
print(predictors.shape)
print(Z.shape)
print(X.shape)
print(y.shape)
print(np.linalg.matrix_rank(X))


(125, 3)
(125, 3)
(125, 4)
(125, 1)
4


Predictors is a 125x3 matrix, Z is a 125x3 matrix, X is a 125x4 matrix, and y is a 125x1 column vector. X has a rank of 4, and contains one intercept column and three predictor columns with 125 observations. 

Written Response:
The goal of building these matricies is to ensure that we can fit a linear model of the form $y=\beta*X +\epsilon$. In order to estimate the $\betas$ we must build $\beta = (X^T*X)X^T*y$, and this matrix operation requires that X and y have compatible shapes. 
The intercept column is included as in order to find the individual $\beta$'s we have to ensure that there is a control that holds all other predictors at 0. The intercept column ensures that. By standarizing the predictors, they now have a mean of approximately 0 and standard deviation of 1 for each predictor. Because we are calculating the means and standard deviations "by hand" with matricies of the same size, we can use broadcasting instead of a slower loop to standarize the data. If the sizes were incompatible, we could not use broadcasting. 

**Part 3**

In [6]:
beta_hat, *_ = np.linalg.lstsq(
    X,
    y,
    rcond = None
)
beta_hat

array([[50.61136   ],
       [13.29471912],
       [ 4.7433565 ],
       [-4.10617123]])

The intercept coefficient was the largest at 50.61136, followed by snowpack_pct_normal at 13.2947, then precip_mm at 4.47433, and finally mean_temp_c at -4.10617. 

This means that holding all other predictors constant, an increase in 1 std of snowpack_pct_normal implies an increase in streamflow_index by 13.2947. A one std increase in precip_mm increases streamflow_index by an estimated 4.474 holding all else constant, and a one std increase in mean_temp_c decreases streamflow_index by 4.10617. 

Snowpack_pct_normal has the largest coefficient of any predictor. 

In [8]:
fitted = X @ beta_hat
residuals = y-fitted
rmse = np.sqrt(
    np.mean(
        residuals ** 2
    )
)
print(residuals.mean())
print(rmse)

-4.121147867408581e-15
7.5135286597381805


X @ beta_hat represents the estimate of the true poulation paramater $\beta$ from the sample. The residual mean should be near 0, which it is, becuase the intercept acts as an adjustment parameter. OLS will over and undershoot the actual response values, so it counteracts that with the intercept which shifts up and down until errors balance out to 0. 
RMSE summarizes the Residual Mean Standard Error, or how far off the models predicted values are vs the observed values. 

X * beta_hat produces a Hadamard Product (or element-wise product) which multiplies matrix elements, whereas X @ beta_hat is matrix multiplication (dot product) of the Matrix X and beta_hat which combines the columns and rows.

**Part 4**

In [9]:
Q, R = np.linalg.qr(X)


In [11]:
print(Q.shape)
print(R.shape)

(125, 4)
(4, 4)


In [12]:
np.allclose(
    Q @ R,
    X
)

True

In [14]:
np.allclose(
    Q.T @ Q,
    np.eye(Q.shape[1])
)

True

In [16]:
U,s,Vt = np.linalg.svd(
    X,
    full_matrices = False
)

In [17]:
print(U.shape)
print(s.shape)
print(Vt.shape)

(125, 4)
(4,)
(4, 4)


In [19]:
X_reconstructed = (
    U
    @ np.diag(s)
    @ Vt
)

In [20]:
np.allclose(
    X_reconstructed,
    X
)

True

In [21]:
print(np.linalg.matrix_rank(X))
print(np.linalg.cond(X))

4
3.2951050664290027


In [23]:
X_raw = np.column_stack([
    np.ones(len(predictors)),
    predictors
])

np.linalg.cond(X_raw)

np.float64(3321.1189367925913)

In [29]:
print(s.max())
print(s.min())
print(s.shape)

15.943050785797375
4.83840437994753
(4,)


1. If Q @ R is approximatley X that means that the SVD worked, and we have decomposed X into smaller compatible matricies. 
2. Q.T @ Q = I verifies that Q is an orthonormal matrix, where each column is orthogonal and has a unit length of 1. 
3. The singular values tell us which directions the value of U move
4. A large condition number tells us how well our matrix is conditioned to changes in the input. A large condition number implies that if there are small changes to the inputs the error in the outputs will be larger. Our standardized matrix X has a condition number of 3.29 which means it is a well conditioned matrix and is less susceptivle to errors in the output given changes to the input.
5. When we ran the unstandardized raw X matrix, we got a condition number of, which is significanlty larger than the standardized condition number of 3.29. 
6. The scales of the different units, cause the condition number to increase in the raw matrix X.  
7. To test if the standardized matrix X shows near redundancy we want to look at the at the rank of X, as well as the determinant of the correlation matrix R where $R=\frac{1}{m}X^{T}X$. Given that the rank of X is 4, and X is a 125x4 matrix, there is no perfect redundancy (as rank = n, for an m x n matrix). The determinant of X is also not near 0, which indicates that there is no high redundancy. 

In [33]:
print(np.linalg.matrix_rank(X))
print(X.shape)
R = (1/125) * X.T @ X
np.linalg.det(R)


4
(125, 4)


np.float64(0.2967673491221258)